# Biochemistry Convolutional Neural Networks Example
In this example we will predict protein Tms from their one hot encodings using a CNN

# Install Packages
Only run if you do not have packages installed

In [ ]:
%pip install rdkit
%pip install pandas
%pip install scikit-learn
%pip install matplotlib
%pip install torch torchvision torchaudio #note this will install pytorch for CPU
%pip install optuna

# Import Libraries and Set Hyperparameters
This code imports libraries, sets hyperparameters for data processing and training, sets filenames, and random seed

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import re
import random
from typing import List, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder     

TARGET_COLUMN = "superclass"                
TOP_N_CLASSES = 10
MIN_NUM_PEAKS = 3
MIN_CLASS_SIZE = 20

MZ_MAX = 1000.0
BIN_SIZE = 1.0

BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
RANDOM_SEED = 42

#setting a device allows us to specify to run on cpu or gpu or to let it switch depending on if a gpu is available
#this will run on gpu if available
#because the hyperparameter optimization will be slow, running on gpu will be better
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


#set random seed for all python libraries for preproucibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)

Using device: cpu


# Read and process input files
read input data and make ohe

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim

# Standard 20 amino acids
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

aa_to_idx = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def one_hot_encode_proteins(
    df,
    sequence_col="sequence",
    max_length=None,
    encoding_col="one_hot",
    mask_col="mask"
):
    """
    One-hot encode protein sequences in a pandas DataFrame.

    The one-hot encoding and mask are added as new columns to the dataframe.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing protein sequences.

    sequence_col : str, default="sequence"
        Column containing protein sequences.

    max_length : int or None, default=None
        Maximum sequence length to encode.
        If None, uses the longest sequence in the dataframe.

    encoding_col : str, default="one_hot"
        Name of the column containing the one-hot encoded arrays.

    mask_col : str, default="mask"
        Name of the column containing the masks.

    Returns
    -------
    pandas.DataFrame
        DataFrame with one-hot encoding and mask columns added.
    """

    sequences = df[sequence_col].astype(str).str.upper().tolist()

    # Determine maximum length
    if max_length is None:
        max_length = max(len(seq) for seq in sequences)
    elif not isinstance(max_length, int) or max_length <= 0:
        raise ValueError("max_length must be a positive integer or None.")

    encoded = np.zeros(
        (len(sequences), max_length, len(AMINO_ACIDS)),
        dtype=np.float32
    )

    mask = np.zeros(
        (len(sequences), max_length),
        dtype=bool
    )

    for i, seq in enumerate(sequences):

        # Truncate sequence if necessary
        seq = seq[:max_length]

        for j, aa in enumerate(seq):

            if aa not in aa_to_idx:
                raise ValueError(
                    f"Unknown amino acid '{aa}' in sequence {i} "
                    f"at position {j}"
                )

            encoded[i, j, aa_to_idx[aa]] = 1.0

        # Mark actual sequence positions
        mask[i, :len(seq)] = True
    #note that in our previous version of this method we flattened the ohe, we don't want to do this for a CNN
    #instead we want the matrix to have a shape of 20 x max_length by transposing it
    encoded = encoded.transpose(0, 2, 1)
    # Add each sequence's array to the corresponding dataframe row
    df[encoding_col] = list(encoded)
    df[mask_col] = list(mask)

    return df

df = pd.read_csv("full_promelt_seq.csv")
#remove rows with invalid amino acids
df = df[df["sequence"].str.count("X")==0]
df = df[df["sequence"].str.count("U")==0]
df = df[df["sequence"].str.count("Z")==0]
df = df[df["sequence"].str.count("B")==0]

#remove rows with very long sequences
df = df[df["Length"]<800]

df = one_hot_encode_proteins(df,max_length=800)
df['stable'] = df["Tm"] > 55
df

#prepare data for pytorch
X = np.stack(df["one_hot"].values)  
y = df["stable"].values.astype(np.float32)

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

#split data
#note we will call the split the train/validation split because we will be using the validation data to monitor
#the progress of the neural network's training
#if you want a true test set that could also be split off at this time
X_train, X_val, y_train, y_val = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Set up and train CNN
Try adjusting the numver of filters or layers in the CNN and see how it affects performance

In [3]:
from sklearn.metrics import root_mean_squared_error, r2_score
# =========================
# 1D CNN
# =========================

class ProteinOHECNN(nn.Module):
    def __init__(self, input_length):
        super().__init__()
        #we have 20 channels, one for each amino acid so the input size should be 20
        self.features = nn.Sequential(
            nn.Conv1d(20, 32, kernel_size=9, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=9, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=9, padding=4),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
        )

        pooled_length = input_length // 8
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * pooled_length, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# =========================
# Training / evaluation
# =========================

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for Xb, yb in loader:
        #this ensures our tensors are set up properly to run on gpu or cpu
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(Xb).squeeze(1)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * Xb.size(0)

    return total_loss / len(loader.dataset)

#note this is a decorator, it will make torch.no_grad() apply to the whole function that follows
#this is an alternative to using "with torch.no_grad():" we had in the last example
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    y_true = []
    y_pred = []

    for Xb, yb in loader:
        Xb = Xb.to(device)
        preds = model(Xb)

        y_true.extend(yb.numpy())
        y_pred.extend(preds)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    rmse = root_mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, r2, y_true, y_pred


input_length = 800

model = ProteinOHECNN(input_length=input_length).to(DEVICE)

class_counts = np.bincount(y_train)
#set class weights to improve performance on rare classes
class_weights = len(y_train) / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    train_rmse, train_r2, _, _ = evaluate(model, train_loader, DEVICE)
    val_rmse, val_r2, y_true, y_pred = evaluate(model, val_loader, DEVICE)

    print(
        f"Epoch {epoch:02d} | "
        f"loss={train_loss:.4f} | "
        f"train_rmse={train_rmse:.4f} | train_bal_r2={train_r2:.4f} | "
        f"val_rmse={val_rmse:.4f} | val_bal_r2={val_r2:.4f}"
    )




Epoch 01 | loss=0.4780 | train_rmse=0.4051 | train_bal_r2=0.1562 | val_rmse=0.4068 | val_bal_r2=0.1502
Epoch 02 | loss=0.1747 | train_rmse=0.4028 | train_bal_r2=0.1658 | val_rmse=0.4037 | val_bal_r2=0.1627
Epoch 03 | loss=0.1709 | train_rmse=0.4089 | train_bal_r2=0.1404 | val_rmse=0.4099 | val_bal_r2=0.1371
Epoch 04 | loss=0.1705 | train_rmse=0.4082 | train_bal_r2=0.1432 | val_rmse=0.4101 | val_bal_r2=0.1360
Epoch 05 | loss=0.1678 | train_rmse=0.4102 | train_bal_r2=0.1351 | val_rmse=0.4122 | val_bal_r2=0.1272
Epoch 06 | loss=0.1667 | train_rmse=0.4192 | train_bal_r2=0.0965 | val_rmse=0.4287 | val_bal_r2=0.0559
Epoch 07 | loss=0.1656 | train_rmse=0.4099 | train_bal_r2=0.1364 | val_rmse=0.4131 | val_bal_r2=0.1234
Epoch 08 | loss=0.1633 | train_rmse=0.3897 | train_bal_r2=0.2192 | val_rmse=0.3993 | val_bal_r2=0.1811
Epoch 09 | loss=0.1626 | train_rmse=0.3962 | train_bal_r2=0.1929 | val_rmse=0.4053 | val_bal_r2=0.1564
Epoch 10 | loss=0.1603 | train_rmse=0.3741 | train_bal_r2=0.2805 | val_rm

# Optimizing with Optuna
We will now look at how to optimize hyperparameters with optuna

In [ ]:
import optuna
import copy
import os
import shutil


#this method lets us generate a list of channels with different growth strateiges
def generate_channels(n_layers, initial_channels, channel_growth):

    channels = []

    for i in range(n_layers):

        if channel_growth == "constant":
            n_channels = initial_channels

        elif channel_growth == "double":
            n_channels = initial_channels * (2 ** i)

        elif channel_growth == "add_32":
            n_channels = initial_channels + 32 * i

        else:
            raise ValueError(
                f"Unknown channel growth strategy: {channel_growth}"
            )

        channels.append(n_channels)

    return channels

#we will define our CNN again but this time with more hyparamters as arguments
class MassSpecCNN(nn.Module):
    def __init__(self, input_length, channels=[32,64,128],kernel_size=9,hidden_size=256,dropout=0.3):
        super().__init__()
        
        #this will be a list of our layers
        layers = []
        in_channels = 20
        #iterate through our list of channel sizes and add one conv layer with that channel output
        for out_channels in channels:
            layers.extend([
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=kernel_size,
                    padding=kernel_size // 2
                ),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
                nn.MaxPool1d(2)
            ])

            in_channels = out_channels
        
        #now we turn our layer list into the features part of our network
        self.features = nn.Sequential(*layers)
        
        # We need to calculate the flattened size for our specific CNN to know what the linear input size is
        # to do this we can run dummy features through the network
        with torch.no_grad():
            dummy_input = torch.zeros(
                1, 20, input_length
            )

            # Temporarily set to eval mode for BatchNorm
            self.features.eval()
            dummy_output = self.features(dummy_input)
            self.features.train()

            flattened_size = dummy_output.flatten(1).shape[1]
            
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    

#define an optimization process for Optuna
def objective(trial):

    # ---------------------------------
    # Architecture hyperparameters
    # ---------------------------------
    
    #the suggest functions suggest what initial values Optuna should look at
    #suggest_int chooses integers from a range
    #suggest_categorical chooses one of the options in th elist
    
    #number of layers in the CNN
    n_layers = trial.suggest_int(
        "n_layers", 1, 3
    )

    initial_channels = trial.suggest_categorical(
        "initial_channels", [16, 32, 64]
    )

    # How channels change between layers
    # this will let us allow the same number of channels at each convolution or for them to grow following a set scheme
    channel_growth = trial.suggest_categorical(
        "channel_growth",
        ["constant", "double", "add_32"]
    )
    
    # Construct channel list
    channels = generate_channels(n_layers, initial_channels, channel_growth)

    kernel_size = trial.suggest_categorical(
        "kernel_size", [3, 5, 7]
    )

    hidden_size = trial.suggest_categorical(
        "hidden_size", [32, 64, 128, 256]
    )

    dropout = trial.suggest_float(
        "dropout", 0.0, 0.5
    )

    # ---------------------------------
    # Training hyperparameters
    # ---------------------------------
    #samples from this range on a log scale
    learning_rate = trial.suggest_float(
        "learning_rate", 1e-5, 1e-2, log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay", 1e-6, 1e-2, log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size", [32, 64, 128]
    )

    # ---------------------------------
    # Build model
    # ---------------------------------

    model = MassSpecCNN(
        input_length=input_length,
        channels=channels,
        kernel_size=kernel_size,
        hidden_size=hidden_size,
        dropout=dropout
    ).to(DEVICE)

    # ---------------------------------
    # Create optimizer
    # ---------------------------------
    #we will also use a new hyperparameter for the optimizer here
    # it is weight decay, a regularization penalty that makes weights smaller
    # it is recommended to use a variation of Adam, AdamW when using weight decay
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    criterion = nn.MSELoss()

    # ---------------------------------
    # Training loop
    # ---------------------------------

    best_rmse = 1000

    for epoch in range(20):

        model.train()

        for X, y in train_loader:

            X = X.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()

            predictions = model(X).squeeze(1)

            loss = criterion(predictions, y)

            loss.backward()
            optimizer.step()

        # ---------------------------------
        # Validation
        # ---------------------------------

        model.eval()

        all_preds = []
        all_true = []
        with torch.no_grad():

            for X, y in val_loader:

                X = X.to(DEVICE)
                y = y.to(DEVICE)

                predictions = model(X)
                all_preds.extend(predictions)
                all_true.extend(y)


        rmse = root_mean_squared_error(all_true, all_preds)

        if rmse < best_rmse:
            best_rmse = rmse
            epoch_of_best_rmse = epoch
            best_model_state = copy.deepcopy(model.state_dict())
            


        # Report progress for pruning
        trial.report(rmse, epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()
    

    trial.set_user_attr("best_epoch", epoch_of_best_rmse)
    # Save the weights from the best epoch in this trial
    torch.save(
        best_model_state,
        f"trial_{trial.number}.pth"
    )
    return best_rmse


#run Optuna
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(n_startup_trials=10,multivariate=True,seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=5
    )
)

study.optimize(objective, n_trials=25)
best_trial = study.best_trial
print("Best trial:", best_trial.number)
print("Best accuracy:", best_trial.value)
print("Best epoch:", best_trial.user_attrs["best_epoch"])

[I 2026-10-11 22:25:59,162] A new study created in memory with name: no-name-cac86a37-14e3-4363-ba5f-edd87a767c5f
[I 2026-10-11 22:36:53,237] Trial 0 finished with value: 0.4107196901327333 and parameters: {'n_layers': 2, 'initial_channels': 16, 'channel_growth': 'constant', 'kernel_size': 3, 'hidden_size': 64, 'dropout': 0.09091248360355031, 'learning_rate': 3.5498788321965036e-05, 'weight_decay': 1.6480446427978994e-05, 'batch_size': 32}. Best is trial 0 with value: 0.4107196901327333.
[I 2026-10-11 23:49:43,720] Trial 1 finished with value: 0.4023590045354516 and parameters: {'n_layers': 2, 'initial_channels': 64, 'channel_growth': 'double', 'kernel_size': 5, 'hidden_size': 256, 'dropout': 0.4828160165372797, 'learning_rate': 0.002661901888489057, 'weight_decay': 1.6536937182824424e-05, 'batch_size': 64}. Best is trial 1 with value: 0.4023590045354516.
[I 2026-10-12 00:04:32,245] Trial 2 finished with value: 0.4412490913200785 and parameters: {'n_layers': 1, 'initial_channels': 64, 